# Phase 3 + 4 — BCPO Feature Selection + Bayesian-Optimized SNN

### What's new here vs. the earlier version
Hyperparameters (hidden units, membrane decay `beta`, learning rate, simulated timesteps) were
previously fixed by hand. This version searches them with **Bayesian Optimization** (Optuna's
TPE sampler) — each trial's SNN is trained on `algae_dataset_train.csv` and scored on
`algae_dataset_val.csv` (109 real rows, never used for weight updates). The winning
configuration is then trained once more on the full training set and evaluated **exactly once**
on `algae_dataset_test.csv` (5,340 real rows) — that final number is the one that goes in results.

### Why validation is small and that's OK here
109 rows is enough to *rank* hyperparameter candidates against each other, even if any single
score has some noise — Optuna averages that out across many trials. It would not be enough for a
final reported metric, which is exactly why test (5,340 rows) is kept separate and untouched until
the very end.


In [ ]:
from google.colab import files
import os
import shutil

# Upload the 3 CSV files
uploaded = files.upload()

# Expected filenames
required_files = [
    "algae_dataset_train.csv",
    "algae_dataset_val.csv",
    "algae_dataset_test.csv"
]

# Check that all required files were uploaded
for filename in required_files:
    if filename in uploaded:
        print(f"Uploaded: {filename}")
    else:
        print(f"Missing: {filename}")

# Verify files exist in /content
print("\nFiles in /content:")
for filename in required_files:
    path = f"/content/{filename}"
    if os.path.exists(path):
        print(f"✓ {path}")

## 1. Config

In [ ]:
TRAIN_CSV = '/content/algae_dataset_train.csv'
VAL_CSV   = '/content/algae_dataset_val.csv'
TEST_CSV  = '/content/algae_dataset_test.csv'
RANDOM_SEED = 42

# BCPO (feature selection)
BCPO_N_AGENTS = 20
BCPO_N_ITER   = 40
BCPO_ALPHA    = 0.9
BCPO_CV_FOLDS = 3
BCPO_FITNESS_SAMPLE_CAP = 3000

# Bayesian optimization (Optuna)
N_OPTUNA_TRIALS   = 30
TUNING_EPOCHS     = 40     # shorter runs during search -- enough to rank candidates
FINAL_EPOCHS      = 150    # full training run for the winning config

MODEL_OUT_PATH    = '/content/snn_bloom_classifier_tuned.pt'
FEATURES_OUT_PATH = '/content/selected_features.json'
STUDY_OUT_PATH    = '/content/optuna_study_results.csv'

## 2. Install & imports

In [ ]:
!pip install snntorch scikit-learn optuna -q

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import snntorch as snn
from snntorch import surrogate
import optuna
from optuna.samplers import TPESampler

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)   # keep trial-by-trial noise down

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. Load train / validation / test (already split correctly upstream -- no re-splitting here)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

non_feature_cols = ['longitude', 'latitude', 'cyan_dn', 'label', 'source_date', 'is_synthetic']
feature_columns = [c for c in train_df.columns if c not in non_feature_cols]
assert len(feature_columns) == 31

if 'is_synthetic' in train_df.columns:
    n_synth = train_df['is_synthetic'].sum()
    print(f'Train: {len(train_df)} rows ({n_synth} synthetic, {n_synth/len(train_df)*100:.1f}%)')
print(f'Validation: {len(val_df)} rows (100% real)   Test: {len(test_df)} rows (100% real)')

X_train_raw = train_df[feature_columns].values.astype(np.float64)
y_train = train_df['label'].values.astype(np.int64)
X_val_raw = val_df[feature_columns].values.astype(np.float64)
y_val = val_df['label'].values.astype(np.int64)
X_test_raw = test_df[feature_columns].values.astype(np.float64)
y_test = test_df['label'].values.astype(np.int64)

scaler = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

print('Train:', X_train.shape, ' Val:', X_val.shape, ' Test:', X_test.shape)

## 4. BCPO feature selection (unchanged mechanism, fitness on a capped training subsample)

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def binarize(continuous_vector, rng):
    probs = sigmoid(continuous_vector)
    return (rng.random(len(continuous_vector)) < probs).astype(int)

_rng_sub = np.random.default_rng(RANDOM_SEED)
if len(X_train) > BCPO_FITNESS_SAMPLE_CAP:
    sub_idx0 = _rng_sub.choice(np.where(y_train == 0)[0],
                                size=min(BCPO_FITNESS_SAMPLE_CAP // 2, (y_train == 0).sum()), replace=False)
    sub_idx1 = _rng_sub.choice(np.where(y_train == 1)[0],
                                size=min(BCPO_FITNESS_SAMPLE_CAP // 2, (y_train == 1).sum()), replace=False)
    fitness_idx = np.concatenate([sub_idx0, sub_idx1])
else:
    fitness_idx = np.arange(len(X_train))

X_fit, y_fit = X_train[fitness_idx], y_train[fitness_idx]
print('BCPO fitness evaluated on', len(X_fit), 'rows.')

def fitness_function(binary_mask, X=X_fit, y=y_fit, alpha=BCPO_ALPHA, cv=BCPO_CV_FOLDS):
    selected = np.where(binary_mask == 1)[0]
    if len(selected) == 0:
        return 1.0
    clf = KNeighborsClassifier(n_neighbors=3)
    scores = cross_val_score(clf, X[:, selected], y, cv=cv)
    return alpha * (1 - scores.mean()) + (1 - alpha) * (len(selected) / X.shape[1])

class BCPO:
    def __init__(self, n_agents=BCPO_N_AGENTS, n_iter=BCPO_N_ITER, dim=31, seed=RANDOM_SEED):
        self.n_agents, self.n_iter, self.dim = n_agents, n_iter, dim
        self.rng = np.random.default_rng(seed)

    def optimize(self, verbose=True):
        pos = self.rng.uniform(-1, 1, size=(self.n_agents, self.dim))
        fitness = np.array([fitness_function(binarize(pos[i], self.rng)) for i in range(self.n_agents)])
        best_idx = int(np.argmin(fitness))
        best_pos, best_fit = pos[best_idx].copy(), fitness[best_idx]

        for t in range(self.n_iter):
            T = 2 * (1 - t / self.n_iter)
            for i in range(self.n_agents):
                r = self.rng.random()
                a, b = self.rng.integers(0, self.n_agents, size=2)
                if r < 0.25:
                    new_pos = pos[i] + self.rng.standard_normal(self.dim) * (pos[a] - pos[b])
                elif r < 0.5:
                    new_pos = pos[i] + self.rng.random(self.dim) * (pos.mean(axis=0) - pos[i])
                elif r < 0.75:
                    new_pos = best_pos + T * self.rng.standard_normal(self.dim) * (best_pos - pos[i])
                else:
                    new_pos = best_pos + self.rng.random(self.dim) * (best_pos - pos[i]) * 0.5
                new_pos = np.clip(new_pos, -3, 3)
                new_bin = binarize(new_pos, self.rng)
                new_fit = fitness_function(new_bin)
                if new_fit < fitness[i]:
                    pos[i], fitness[i] = new_pos, new_fit
                    if new_fit < best_fit:
                        best_fit, best_pos = new_fit, new_pos.copy()
            if verbose:
                print(f'Iter {t+1:3d}/{self.n_iter}  best fitness={best_fit:.4f}  '
                      f'selected~{binarize(best_pos, self.rng).sum()}')

        best_mask = binarize(best_pos, self.rng)
        if best_mask.sum() == 0:
            best_mask[np.argmax(best_pos)] = 1
        return best_mask, best_fit

optimizer = BCPO(dim=len(feature_columns))
best_mask, best_fit = optimizer.optimize()
selected_idx = np.where(best_mask == 1)[0]
selected_features = [feature_columns[i] for i in selected_idx]
print(f'\nSelected {len(selected_features)}/{len(feature_columns)} features: {selected_features}')

X_train_sel = X_train[:, selected_idx]
X_val_sel   = X_val[:, selected_idx]
X_test_sel  = X_test[:, selected_idx]

## 5. SNN architecture (hyperparameters now come from Optuna, not fixed by hand)

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.fc1  = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2  = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x, num_steps):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk2_rec = []
        for _ in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
        return torch.stack(spk2_rec, dim=0)


def train_and_score(hidden_units, beta, lr, num_steps, epochs, X_tr, y_tr, X_ev, y_ev):
    model = SNNClassifier(len(selected_features), hidden_units, 2, beta).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long, device=device)
    X_ev_t = torch.tensor(X_ev, dtype=torch.float32, device=device)

    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        spk_rec = model(X_tr_t, num_steps)
        loss = loss_fn(spk_rec.sum(dim=0), y_tr_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        spk_rec_ev = model(X_ev_t, num_steps)
        preds = spk_rec_ev.sum(dim=0).argmax(dim=1).cpu().numpy()

    return f1_score(y_ev, preds), model

## 6. Bayesian optimization (Optuna, TPE sampler) — scored on validation only
Each trial trains a full SNN with a candidate hyperparameter set and scores it on the 109-row real
validation set. TPE is a Bayesian method: it builds a probabilistic model of which regions of the
search space look promising based on trials so far, and samples new candidates from that model
rather than searching randomly or exhaustively.

In [ ]:
def objective(trial):
    hidden_units = trial.suggest_int('hidden_units', 8, 64)
    beta = trial.suggest_float('beta', 0.5, 0.99)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    num_steps = trial.suggest_int('num_steps', 10, 40)

    f1, _ = train_and_score(hidden_units, beta, lr, num_steps, TUNING_EPOCHS,
                             X_train_sel, y_train, X_val_sel, y_val)
    return f1

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print('Best validation F1:', study.best_value)
print('Best hyperparameters:', study.best_params)

study.trials_dataframe().to_csv(STUDY_OUT_PATH, index=False)
print('Saved full trial history to', STUDY_OUT_PATH)

## 7. Train the final model with the winning hyperparameters (full epoch budget)

In [ ]:
best = study.best_params
final_f1_on_val, final_model = train_and_score(
    best['hidden_units'], best['beta'], best['lr'], best['num_steps'], FINAL_EPOCHS,
    X_train_sel, y_train, X_val_sel, y_val
)a
print('Final model validation F1 (sanity check, not the reported metric):', final_f1_on_val)

## 8. Evaluate ONCE on the real test set — this is the number that gets reported

In [ ]:
final_model.eval()
X_test_t = torch.tensor(X_test_sel, dtype=torch.float32, device=device)
with torch.no_grad():
    spk_rec_test = final_model(X_test_t, best['num_steps'])
    spk_counts_test = spk_rec_test.sum(dim=0)
    test_preds = spk_counts_test.argmax(dim=1).cpu().numpy()
    test_probs = torch.softmax(spk_counts_test, dim=1)[:, 1].cpu().numpy()

test_acc = accuracy_score(y_test, test_preds)
test_f1  = f1_score(y_test, test_preds)

print('TEST accuracy:', round(test_acc, 4))
print('TEST F1:', round(test_f1, 4))
print()
print(confusion_matrix(y_test, test_preds))
print()
print(classification_report(y_test, test_preds, target_names=['No Bloom', 'Bloom']))

## 9. Save model, hyperparameters, and selected features

In [ ]:
torch.save({
    'model_state_dict': final_model.state_dict(),
    'selected_features': selected_features,
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
    'best_hyperparams': best,
    'config': {
        'num_inputs': len(selected_features),
        'num_hidden': best['hidden_units'],
        'num_outputs': 2,
        'beta': best['beta'],
        'num_steps': best['num_steps'],
    }
}, MODEL_OUT_PATH)

with open(FEATURES_OUT_PATH, 'w') as f:
    json.dump(selected_features, f, indent=2)

print('Saved model to:', MODEL_OUT_PATH)
print('Saved selected features to:', FEATURES_OUT_PATH)
print('Best hyperparameters:', best)

In [ ]:
from google.colab import files

files.download('/content/snn_bloom_classifier_tuned.pt')
files.download('/content/selected_features.json')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# FINAL SNN PREDICTION ERROR DISTRIBUTION
# Based on the actual 5,340 test predictions
# ============================================================

# Actual confusion matrix
TN = 1784
FP = 152
FN = 86
TP = 3318

# Total
total = TN + FP + FN + TP

# ------------------------------------------------------------
# Create category labels from the actual confusion matrix
# ------------------------------------------------------------

categories = np.array(
    ["Correct No-Bloom"] * TN +
    ["False Positive"] * FP +
    ["False Negative"] * FN +
    ["Correct Bloom"] * TP
)

# ------------------------------------------------------------
# Deterministic arrangement
# ------------------------------------------------------------

# Shuffle only for visual distribution.
# This DOES NOT represent geographic coordinates.
rng = np.random.default_rng(42)
rng.shuffle(categories)

# Create a compact grid
n_cols = 100
n_rows = int(np.ceil(total / n_cols))

x = np.arange(total) % n_cols
y = np.arange(total) // n_cols

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(14, 7))

# Plot each category
for category, marker, size, alpha in [
    ("Correct No-Bloom", "o", 10, 0.55),
    ("Correct Bloom", "o", 10, 0.55),
    ("False Positive", "x", 35, 0.95),
    ("False Negative", "x", 35, 0.95)
]:

    mask = categories == category

    plt.scatter(
        x[mask],
        y[mask],
        s=size,
        marker=marker,
        alpha=alpha,
        label=f"{category} ({mask.sum()})"
    )

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

plt.xlabel("Test sample index", fontsize=12)
plt.ylabel("Sample arrangement", fontsize=12)

plt.title(
    "Distribution of Correct and Incorrect Predictions\n"
    "Final SNN — 5,340 Real Held-Out Test Samples",
    fontsize=14,
    fontweight="bold"
)

plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=2,
    frameon=True
)

plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
print("Variables currently available:")
print(sorted([x for x in globals().keys() if not x.startswith("_")]))

## Next: Phase 5 (visualization)
`visualize_results.ipynb` already works with checkpoints in this format — just point `MODEL_PATH`
at `snn_bloom_classifier_tuned.pt` and `TEST_CSV` at `algae_dataset_test.csv`.
